# Spatial Iteration Engine - Panel de Control Completo

Este notebook proporciona acceso completo a todas las funcionalidades del Spatial Iteration Engine:

- **Red**: Configuración de red (Local, Broadcast, Multicast, IP directa)
- **Motor**: Control de inicio/parada y selección de cámara
- **Filtros**: Aplicación de filtros de imagen (Edges, Brightness, Invert, etc.)
- **Vista**: Configuración de renderizado (ASCII/RAW, charset, contraste, brillo)
- **IA**: Percepción en tiempo real (detección de cara, manos, pose)

**Requisitos**:
- Jupyter Notebook o JupyterLab
- ipywidgets: `pip install ipywidgets`
- Cámara conectada (opcional, para pruebas)

**Nota**: Este notebook debe ejecutarse desde la raíz del repositorio `Spatial-Iteration-Engine`


## 1. Configuración Inicial

Configura las rutas y variables de entorno necesarias.


In [ ]:
import os
import sys
from pathlib import Path

# Obtener la raíz del repositorio (donde está este notebook)
# Si ejecutas desde Spatial-Iteration-Engine/, esto debería funcionar
repo_root = Path.cwd()

# Verificar que estamos en la raíz correcta
if not (repo_root / "python" / "ascii_stream_engine").exists():
    # Intentar desde el directorio del notebook
    repo_root = Path(__file__).parent.parent.parent if '__file__' in globals() else Path.cwd().parent.parent
    if not (repo_root / "python" / "ascii_stream_engine").exists():
        raise RuntimeError(
            f"No se encontró el directorio python/ascii_stream_engine. "
            f"Asegúrate de ejecutar este notebook desde la raíz del repositorio. "
            f"Directorio actual: {Path.cwd()}"
        )

print(f"✅ Repositorio raíz: {repo_root}")

# Configurar PYTHONPATH
python_path = str(repo_root / "python")
cpp_build_path = str(repo_root / "cpp" / "build")

if python_path not in sys.path:
    sys.path.insert(0, python_path)
if cpp_build_path not in sys.path:
    sys.path.insert(0, cpp_build_path)

print(f"✅ PYTHONPATH configurado")
print(f"   - Python: {python_path}")
print(f"   - C++ Build: {cpp_build_path}")

# Configurar variable de entorno para modelos ONNX
onnx_models_dir = repo_root / "onnx_models" / "mediapipe"
os.environ["ONNX_MODELS_DIR"] = str(onnx_models_dir)

print(f"✅ ONNX_MODELS_DIR: {onnx_models_dir}")

# Verificar que los modelos existan
models = ["face_landmark.onnx", "hand_landmark.onnx", "pose_landmark.onnx"]
for model in models:
    model_path = onnx_models_dir / model
    if model_path.exists():
        size_mb = model_path.stat().st_size / (1024 * 1024)
        print(f"   ✅ {model} ({size_mb:.1f} MB)")
    else:
        print(f"   ⚠️  {model} no encontrado")


## 2. Importar Módulos

Importa todas las clases y funciones necesarias.


In [ ]:
try:
    from ascii_stream_engine.presentation.notebook_api import (
        build_engine_for_notebook,
        build_general_control_panel,
        build_diagnostics_panel,
    )
    print("✅ Módulos de presentación importados")
except ImportError as e:
    print(f"❌ Error importando módulos: {e}")
    print("   Asegúrate de que PYTHONPATH esté configurado correctamente")
    raise

try:
    import perception_cpp
    print("✅ Módulo C++ perception_cpp disponible")
except ImportError:
    print("⚠️  Módulo C++ perception_cpp no disponible (percepción IA no funcionará)")
    print("   Compila con: cd cpp/build && cmake .. && make")

print("\n✅ Todos los módulos importados correctamente")


## 3. Crear Motor con Percepción IA

Crea el StreamEngine con soporte completo de percepción (cara, manos, pose).


In [ ]:
# Crear el motor con cámara (índice 0 por defecto)
# Puedes cambiar el índice si tienes múltiples cámaras
camera_index = 0

print(f"📷 Creando motor con cámara índice {camera_index}...")

try:
    engine = build_engine_for_notebook(camera_index=camera_index)
    print("✅ Motor creado exitosamente")
    
    # Verificar que los analyzers estén disponibles
    if hasattr(engine, "_analyzers") and engine._analyzers:
        num_analyzers = len(engine._analyzers.analyzers) if hasattr(engine._analyzers, "analyzers") else 0
        print(f"✅ Pipeline de percepción: {num_analyzers} analyzers disponibles")
        for analyzer in getattr(engine._analyzers, "analyzers", []):
            print(f"   - {analyzer.name}: {'✅' if analyzer.enabled else '❌'}")
    else:
        print("⚠️  Pipeline de percepción no disponible")
        
except Exception as e:
    print(f"❌ Error creando motor: {e}")
    import traceback
    traceback.print_exc()
    raise


## 4. Panel de Control Completo

Muestra el panel de control con todas las funcionalidades:

### Pestañas disponibles:

1. **Red**: Configuración de red (modo, host, puerto)
2. **Motor**: Control de inicio/parada y cámara
3. **Filtros**: Aplicación de filtros de imagen
4. **Vista**: Configuración de renderizado (ASCII/RAW)
5. **IA**: Percepción en tiempo real (cara, manos, pose)

### Instrucciones de uso:

1. Configura las opciones en cada pestaña
2. Para IA: Activa las detecciones deseadas y selecciona "Overlay landmarks" si quieres ver los puntos
3. Haz clic en "Aplicar" en cada pestaña para aplicar los cambios
4. Presiona "Start" en la pestaña Motor para iniciar el stream
5. Usa "Actualizar estado detector" en la pestaña IA para ver cuántos puntos se detectan


In [ ]:
# Crear y mostrar el panel de control completo
control_panel = build_general_control_panel(engine)

print("\n✅ Panel de control mostrado")
print("\n📋 Pestañas disponibles:")
print("   1. Red - Configuración de red")
print("   2. Motor - Control de inicio/parada")
print("   3. Filtros - Filtros de imagen")
print("   4. Vista - Configuración de renderizado")
print("   5. IA - Percepción en tiempo real")
print("\n💡 Usa las pestañas para configurar y luego presiona 'Start' en Motor")


## 5. Panel de Diagnósticos (Opcional)

Muestra información de diagnóstico del sistema y del motor.


In [ ]:
# Crear y mostrar el panel de diagnósticos
diagnostics = build_diagnostics_panel(engine)

print("✅ Panel de diagnósticos mostrado")
print("\n📊 Información disponible:")
print("   - Estado del motor")
print("   - Métricas de performance")
print("   - Información del sistema")
print("   - Estado de los módulos")


## 6. Notas y Troubleshooting

### Problemas Comunes

1. **Error: No module named 'perception_cpp'**
   - Solución: Compila el módulo C++: `cd cpp/build && cmake .. && make`

2. **Error: Model not found**
   - Solución: Verifica que `ONNX_MODELS_DIR` esté configurado y los modelos existan

3. **No se detectan puntos en IA**
   - Verifica que los analyzers estén habilitados en la pestaña IA
   - Asegúrate de que el motor esté corriendo
   - Verifica que haya una persona/cara visible en la cámara

4. **La cámara no funciona**
   - Verifica que la cámara esté conectada
   - Prueba cambiar el índice de cámara (0, 1, 2, etc.)

### Recursos

- Documentación: `docs/PERCEPTION_INTEGRATION.md`
- Scripts de prueba: `scripts/test_perception_integration.py`
- Modelos: `onnx_models/mediapipe/`
